# Which argument actually moved the debate?

An argument-sensitivity run for [Lightningfish](https://github.com/rajul-kk/LightningFish), on Kaggle's free T4 GPU — same Ollama/qwen2.5:7b setup as the other notebooks here, at the same population size (24 agents, 4 rounds).

This is a different kind of question from the backtests elsewhere in this repo. Bounded confidence and the co-evolving network were both tested against real settled outcomes and found to be non-effects on the controversy axis — see METHODOLOGY.md. Counterfactual replay (`excluded_argument_tags`, added in the same round of work) can't be tested that way at all: there's no ground truth for "what would have happened if this argument had never been raised," because HN only gives us the timeline where it *was* raised.

What it can do instead needs no ground truth: for one already-run simulation, re-run it once per argument tag with that tag excluded from circulation, and see how far the final opinion moves without it. That's `argument_sensitivity_report()` — a before/after comparison against the simulation's own baseline, not a prediction scored against reality. This notebook runs it on a handful of real, cached HN stories at full population size, something a CPU-only smoke test (4 agents, 1 round) already confirmed works end to end — but at that size the result was dominated by noise, which is exactly why this needs the GPU.

## The noise floor, and why it matters here

The smoke test surfaced something worth knowing before reading any output below: even a tag that never got posted in the baseline run showed a tiny nonzero delta when excluded. `random.seed()` is reset before every arm specifically to kill one source of that — follower-graph sampling — but the LLM itself is still stochastic, so two runs of the *same* prompts don't reproduce byte-identical trajectories. A delta comparable in size to that noise floor isn't a demonstrated effect; only one that clearly stands out from it is. This notebook doesn't estimate the floor directly (that would cost as much as the whole sweep again), so read small deltas skeptically and large, direction-flipping ones as the interesting cases.


---
## 1. Setup

Sidebar: **Accelerator → GPU**, **Internet → On**.

Install `zstd` before Ollama — its installer needs it to extract, Kaggle's base image doesn't ship it, and skipping this makes the install fail quietly, surfacing later as a confusing `FileNotFoundError: 'ollama'`.


In [ ]:
!apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y zstd > /dev/null 2>&1
!zstd --version || echo "WARNING: zstd missing - the install below will fail"
!curl -fsSL https://ollama.com/install.sh | sh


In [ ]:
import shutil, subprocess, time, requests

if shutil.which("ollama") is None:
    raise RuntimeError(
        "ollama not found after install. Scroll up for the installer's error: "
        "usually zstd (cell above) or Internet disabled in the sidebar."
    )

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(60):
    try:
        requests.get("http://localhost:11434/api/tags", timeout=2)
        print("ollama up"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("ollama installed but the server did not start")


In [ ]:
MODEL = "qwen2.5:7b"

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!ollama pull {MODEL}

requests.post("http://localhost:11434/api/generate",
              json={"model": MODEL, "prompt": "hi", "stream": False, "keep_alive": -1},
              timeout=600)

for m in requests.get("http://localhost:11434/api/ps").json().get("models", []):
    vram = m.get("size_vram", 0) / 1e9
    print(f"{m['name']}: {vram:.2f} GB in VRAM")
    assert vram > 0, "model landed on CPU - enable the GPU accelerator, this is pointless otherwise"
print("GPU inference confirmed")


In [ ]:
!git clone --depth 1 https://github.com/rajul-kk/LightningFish.git /kaggle/working/lf
!pip -q install anthropic openai scipy requests pytest praw yfinance

import os, sys
os.chdir("/kaggle/working/lf")
sys.path.insert(0, "/kaggle/working/lf")

# Engine + HN suites only; also confirms argument_sensitivity.py and its
# tests are actually present in this clone.
!python -m pytest tests/core tests/hn -q 2>&1 | tail -5


---
## 2. Prime the event cache

`run_argument_sensitivity.py` reads real seeds from the local event cache — it doesn't hit the network itself. So the first step is just pulling a handful of real HN stories into that cache, the same way any of the backtests do, except this doesn't need to simulate or score anything yet.


In [ ]:
from lightningfish_core.event_cache import EventCache, cached_pull_events
from lightningfish_hn.backtest_events import pull_hn_events

PRIME_LIMIT = 20  # stories to have on hand; only SENSITIVITY_LIMIT below get analyzed

cache = EventCache("hn_stories")
events = cached_pull_events(cache, f"hn:points:{PRIME_LIMIT}", lambda: pull_hn_events("points", PRIME_LIMIT))
print(f"{len(events)} stories cached and ready")


---
## 3. Configuration

Cost here is `events × (len(taxonomy) + 1)` simulations — HN's taxonomy has 8 tags, so 9 simulations per event. `SENSITIVITY_LIMIT=4` keeps this to 36 simulations total, comparable to a small backtest run.


In [ ]:
N_AGENTS = 24   # same population size as the other Kaggle notebooks here
N_ROUNDS = 4
SENSITIVITY_LIMIT = 4   # events to actually analyze, out of the ones primed above

os.environ["LIGHTNINGFISH_MODEL"] = f"ollama:{MODEL}"
os.environ["LIGHTNINGFISH_N_AGENTS"] = str(N_AGENTS)
os.environ["LIGHTNINGFISH_N_ROUNDS"] = str(N_ROUNDS)
os.environ["LIGHTNINGFISH_SENSITIVITY_LIMIT"] = str(SENSITIVITY_LIMIT)
os.environ["LIGHTNINGFISH_LOCAL_TIMEOUT"] = "120"
os.environ["PYTHONUNBUFFERED"] = "1"
print(f"{MODEL} | {N_AGENTS} agents x {N_ROUNDS} rounds | analyzing {SENSITIVITY_LIMIT} of {PRIME_LIMIT} cached events")
print(f"cost: {SENSITIVITY_LIMIT} events x 9 simulations = {SENSITIVITY_LIMIT * 9} total runs")


### Throughput check

Each simulation here is the same shape as a backtest event (~26 model calls), just 9x as many per event. Worth confirming the per-call cost before committing.


In [ ]:
import time
from lightningfish_core.llm_provider import make_provider

provider = make_provider(f"ollama:{MODEL}")
t0 = time.time()
for _ in range(3):
    provider.get_opinion("Output ONLY a number between -1 and 1.", "Rate: 0.5", f"ollama:{MODEL}")
per_call = (time.time() - t0) / 3
per_event = per_call * 26
print(f"{per_call:.2f}s per call  ->  ~{per_event:.0f}s per simulation  "
      f"->  ~{per_event * 9 * SENSITIVITY_LIMIT / 60:.1f} min total")


---
## 4. Run it


In [ ]:
!python -m tests.integration.run_argument_sensitivity hn 2>&1 | tee /kaggle/working/argument_sensitivity.log


### Reading the output

Each event prints a baseline mean opinion and direction, then one line per taxonomy tag showing what the mean would have been without it, the delta, and whether removing it would have flipped the predicted direction. A tag marked "(never posted this run)" means nobody actually used that tag in the baseline — excluding it should show a delta near the noise floor described in the intro, and a genuinely large delta there would be a red flag worth investigating rather than trusting.

The interesting rows are the ones with a delta that's both large relative to the others in the same event *and* attached to a tag that actually got posted. `<-- flips the call` is the strongest possible signal — an argument whose absence would have changed the simulation's prediction entirely. Across only 4 events at n=24, don't expect these to be dramatic every time; the honest read is which arguments *tend* to show up as high-leverage across the sample, not a definitive ranking from one run.


---
## 5. Save


In [ ]:
!cp -r .cache/lightningfish /kaggle/working/cache
!ls -la /kaggle/working/*.log /kaggle/working/cache


The log and the primed event cache are both saved to `/kaggle/working/` — download from the notebook's Output tab. Re-running `run_argument_sensitivity.py` against the same cache later costs nothing extra to re-prime, though it will re-simulate (there's no run-level cache for this script the way the backtests have one, since every arm here is already a fresh simulation by design).
